# 14 · RL with a larger KL penalty (`grpo_v4`, `grpo_v5`)

**Why.** In notebook 10, adding a KL penalty (β = 0.04) to RL clearly helped: `grpo_v2` solved significantly more puzzles than `grpo_v1` (4.9% vs. 3.3%), and its share of pawn moves fell from 41% to 25% of its legal answers. But it is still significantly below `sft_v1` (7.8%), and the KL divergence from the SFT model kept rising during training (from 0 to about 0.29). So β = 0.04 slowed the drift away from SFT without stopping it. This notebook asks: **does a larger penalty keep the model closer to SFT, and does that keep its puzzle accuracy?**

**The KL penalty, briefly.** The *KL divergence* measures how far the RL model's probabilities over answers have moved from those of the SFT model. GRPO subtracts β × KL from what it maximizes, so β sets the price of moving away from SFT. With a larger β, the model changes only where the reward gain is larger.

**What changes.** Two runs, each starting from `sft_v1`, with everything else exactly as in `grpo_v2` (the original reward, the same 2,000 positions, 8 answers per position, learning rate, LoRA):

| | `grpo_v1` | `grpo_v2` | `grpo_v4` | `grpo_v5` |
|---|---|---|---|---|
| KL penalty β | 0 | 0.04 | **0.1** | **0.2** |

Together with `grpo_v1` and `grpo_v2`, this gives four values of β, from no penalty to a strong one. `grpo_v3`'s reward change made no significant difference in notebook 10, so these runs keep the original reward, and each differs from `grpo_v2` only in β.

**Running it on Databricks**
1. Open this notebook in the shared Git folder and pick **Serverless GPU → Small (1×A10)**. Plain *Serverless* has no GPU. Medium and Large cost much more and are not needed here.
2. Set `QUICK_CHECK = True` in the Settings cell and click **Run all**. This takes a few minutes, uses tiny sizes, and saves nothing to `results/`. It catches setup problems before the long run. (The first cell restarts Python once; if Run all stops there, click Run all again.)
3. Set `QUICK_CHECK = False` and run the real thing as a **Job**, so the browser can close: New → Job → type *Notebook* → this notebook → compute *Serverless GPU, 1×A10*; add an email notification. Separate Jobs can run at the same time.
4. Afterwards, commit and push **only `results/`** from the Git folder. Leave `notebooks/` unticked: Databricks cannot save notebook outputs to git, so committing a notebook from there would erase its outputs on GitHub. The outputs of a Job run are on the Job's run page; every figure and table is also saved in `results/figures/`.

**Time:** about 45 minutes of training and 5 minutes of grading per run, so about 1.5–2 hours in total (roughly $8–10 of credits).

**Outputs**
- the adapters in `checkpoints/grpo_v4/` and `checkpoints/grpo_v5/` (not on GitHub)
- results: `results/grpo_v4*.json`, `results/grpo_v5*.json`, and new rows in `results/summary.csv`
- figures and tables in `results/figures/grpo_v4/` and `results/figures/grpo_v5/`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
QUICK_CHECK = False   # True = a few-minute run with tiny sizes, to check that everything works. Saves nothing to results/.
RUNS = {              # run name -> its changes to configs/rl.yaml
    "grpo_v4": {"training": {"beta": 0.1}},
    "grpo_v5": {"training": {"beta": 0.2}},
}

import copy
import gc

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.width", 200)  # print wide tables on one line

from chesslm import experiments, plots
from chesslm.config import REPO_ROOT, load_config
from chesslm.evaluate import load_eval_model
from chesslm.experiments import (compare_with, figures_dir, grade_puzzles, load_graded, load_training_log, move_kinds,
                                 puzzle_table, save_table, save_training)
from chesslm.rewards import to_rl_dataset
from chesslm.train_rl import train

sft_cfg, prompt_cfg, eval_cfg = load_config("sft"), load_config("prompt"), load_config("eval")
rl_cfg = load_config("rl")
configs = {"model": sft_cfg["model"], "prompt": prompt_cfg, "eval": eval_cfg}
START_ADAPTER = REPO_ROOT / rl_cfg["training"]["start_adapter"]   # sft_v1

if QUICK_CHECK:
    experiments.use_quick_check_folder()
    RUNS = {f"{name}_check": change for name, change in RUNS.items()}
    rl_cfg["training"].update(num_positions=8, logging_steps=1)
PUZZLE_LIMIT = 5 if QUICK_CHECK else None
print(f"Runs: {list(RUNS)} | starting from {START_ADAPTER}")

## 1. Training positions

The same 2,000 training positions as `grpo_v1` and `grpo_v2` (the same seeded sample), with the usual prompt.

In [ ]:
t = rl_cfg["training"]
train_positions = pd.read_parquet(REPO_ROOT / "data" / "processed" / "train.parquet")
train_ds = to_rl_dataset(train_positions.sample(n=t["num_positions"], random_state=t["seed"]), prompt_cfg)
print(f"{len(train_ds):,} positions x {t['num_generations']} answers each = "
      f"{len(train_ds) * t['num_generations']:,} answers scored by Stockfish")

## 2. Train and grade `grpo_v4` and `grpo_v5`

For each run, the average reward and the KL divergence from the SFT model over training. With a larger β, the KL divergence should stay lower than `grpo_v2`'s; the reward may rise more slowly. Each model is then graded on the dev and test puzzles exactly like every other model.

In [ ]:
for run_name, change in RUNS.items():
    cfg = copy.deepcopy(rl_cfg)
    for section, values in change.items():
        cfg[section].update(values)
    adapter_dir = REPO_ROOT / "checkpoints" / run_name
    fig = figures_dir(run_name)
    print(f"=== {run_name}: beta {cfg['training']['beta']} | reward {cfg['reward']} ===")

    log, compute = train(train_ds, cfg, eval_cfg["stockfish"], sft_cfg["model"], sft_cfg["lora"], START_ADAPTER,
                         adapter_dir)
    save_training(run_name, {"rl": cfg, "prompt": prompt_cfg}, compute, log, start_adapter=START_ADAPTER)
    rewards = log.reward.dropna()
    print(f"Average reward: {rewards.iloc[0]:.3f} at the start -> {rewards.iloc[-1]:.3f} at the end")
    plots.reward_curve(log, fig / "reward.png", title=f"{run_name}: average reward of the sampled answers")
    if "kl" in log:
        plots.training_curve(log, "kl", "KL divergence from the SFT model", fig / "kl.png",
                             title=f"{run_name}: how far the model moves from SFT")
    plt.show()

    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_eval_model(sft_cfg["model"], adapter_path=adapter_dir)
    grade_puzzles(model, tokenizer, "dev", f"{run_name}_dev", configs, str(adapter_dir), PUZZLE_LIMIT)
    grade_puzzles(model, tokenizer, "test", run_name, configs, str(adapter_dir), PUZZLE_LIMIT)
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 3. Results: four values of β

**How to read this section**
- **The training curves** put all runs with a KL penalty on one figure: β = 0.04 (`grpo_v2`), 0.1 and 0.2. A larger β should give a lower KL curve.
- **Puzzle accuracy** is the main metric; `sft_v1` (7.8%) is the line to beat, and a random legal move solves 5.6%. If accuracy rises with β towards `sft_v1`, the penalty is protecting what SFT learned.
- **What kind of moves** each model plays, among its legal test answers: the share of pawn moves, captures, and checks. `grpo_v1`'s failure was a shift to pawn moves (41%, vs. 2% for `sft_v1`).
- The **paired comparisons** count puzzles solved by only one of two models; a p-value below 0.05 means the difference is unlikely to be chance.

All numbers are on the 1,000 test puzzles.

In [ ]:
v4, v5 = RUNS
runs = ["sft_v1", "grpo_v1", "grpo_v2", v4, v5]
betas = {"grpo_v2": 0.04, v4: RUNS[v4]["training"]["beta"], v5: RUNS[v5]["training"]["beta"]}
logs = {f"{name} (beta {beta})": load_training_log(name) for name, beta in betas.items()}

summary = pd.read_csv(experiments.RESULTS / "summary.csv")
comparison = pd.DataFrame([compare_with(v4, "sft_v1"), compare_with(v5, "sft_v1"), compare_with(v4, "grpo_v2"),
                           compare_with(v5, "grpo_v2"), compare_with(v5, v4)]).set_index("model")

puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / "puzzles_test.csv").set_index("puzzle_id")
kinds = pd.DataFrame({run: move_kinds(g, puzzles.fen.loc[g.puzzle_id]) for run, g in
                      ((run, load_graded(run)) for run in runs)}).T.round(1)
kinds.columns = ["Pawn moves (%)", "Captures (%)", "Checks (%)"]
kinds.index.name = "Model"

for run_name in RUNS:  # the same comparison is saved in each new run's folder
    fig = figures_dir(run_name)
    table = puzzle_table(runs, fig)
    table.insert(0, "KL penalty beta", [None, 0.0] + list(betas.values()))
    save_table(table, fig, "puzzle_results", "1,000 test puzzles (250 per rating band). Rating-band columns are "
                                             "puzzle accuracy (%) in that band. ACPL is over legal answers only.")
    save_table(comparison, fig, "paired_comparison",
               "Puzzles solved by only one of the two models, and the sign-test p-value (below 0.05 = unlikely to be chance).")
    save_table(kinds, fig, "move_kinds", "Share of each model's legal test answers that are pawn moves, captures, or checks.")
    plt.close("all")  # only the last copy of each figure is shown below
    plots.training_curves(logs, "kl", "KL divergence from the SFT model", fig / "kl_by_beta.png",
                          title="How far each RL model moves from SFT, by KL penalty")
    plots.training_curves(logs, "reward", "average reward of sampled answers", fig / "reward_by_beta.png",
                          title="Average reward during training, by KL penalty")
    plots.compare_metrics(summary, runs, fig / "puzzle_metrics.png", title="SFT and RL with four values of the KL penalty")
    plots.accuracy_by_band(summary, runs, fig / "accuracy_by_band.png", title="Puzzle accuracy by puzzle difficulty")
    plots.outcome_breakdown(summary, runs, fig / "answer_outcomes.png", title="Are the answers legal moves?")
    plots.metric_panels(kinds, [(c, c, "{:.0f}") for c in kinds.columns], fig / "move_kinds.png",
                        title="What kind of moves does each model play? (legal test answers)")

display(table)
display(comparison)
display(kinds)

## Summary of outputs

- `results/figures/grpo_v4/` and `results/figures/grpo_v5/` (the same comparison in both): `reward.png`, `kl.png`, `kl_by_beta.png`, `reward_by_beta.png`, `puzzle_results` (table, with β), `paired_comparison` (table), `move_kinds` (table and figure), `puzzle_metrics.png`, `accuracy_by_band.png`, `answer_outcomes.png`
- `results/grpo_v4.json`, `results/grpo_v5.json`: every test answer and its grade; `*_training.json`: settings, compute, training log
- Both models are also graded on held-out game positions and in full games in notebook 11.
- The best β here is the RL setting used in the combinations (notebook 15).